In [0]:
%pip install folium==0.17.0 plotly==5.24.1 branca==0.7.2 "numpy<2.0" --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 07 — Medical Desert Scoring & Map Visualization (v12 — Full Schema-Compatible Rebuild)
# MAGIC
# MAGIC **Rebuilt against verified schemas:**
# MAGIC - `gold_idp_enriched`           — 190 cols, IDP-enriched arrays, full v12 quality signals
# MAGIC - `gold_facilities_enriched_v12`— 179 cols, v12 quality/risk/geo/graph fields, facility-level desert
# MAGIC - `gold_regional_summary_v12`   — 70 cols, authoritative per-capita metrics + gap scores
# MAGIC
# MAGIC ## Architecture
# MAGIC
# MAGIC ### Part A — Medical Desert Scoring
# MAGIC ```
# MAGIC BLENDED_MDS = 0.40 × base_mds_component      (from gold_regional_summary per-capita metrics)
# MAGIC            + 0.25 × specialty_gap_component   (emergency/ICU/surgery/maternity gap scores)
# MAGIC            + 0.20 × integrity_component        (ghost probability + quality risk + anomalies)
# MAGIC            + 0.15 × confidence_component       (completeness, geo quality, readiness)
# MAGIC ```
# MAGIC Facility-level overlay adds per-facility desert exposure score using v12 risk fields.
# MAGIC
# MAGIC ### Part B — Visualization
# MAGIC - Folium choropleth (region circle markers + facility cluster) — citation-aware popups
# MAGIC - Plotly dashboard: MDS bars, specialty gaps, type dist, doctor/bed scatter,
# MAGIC   component breakdown, anomaly overlay, NGO/volunteer coverage
# MAGIC - Exports: HTML map, chart HTMLs, GeoJSON, CSV
# MAGIC
# MAGIC ### Part C — Persistence
# MAGIC - `gold_medical_desert_scores`     — region-level blended MDS + components + rationale
# MAGIC - `gold_facility_desert_overlay`  — facility-level desert exposure scores
# MAGIC - MDS back-fill into `gold_idp_enriched` and `gold_facilities_enriched`

# COMMAND ----------
# MAGIC %pip install folium==0.17.0 plotly==5.24.1 branca==0.7.2 "numpy<2.0" --quiet

# COMMAND ----------
dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json
import os
import re
import math
from typing import List, Dict, Optional, Any, Tuple
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow

from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.getOrCreate()
print(f"Folium   : {folium.__version__}")
print(f"Run at   : {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------

class MapConfig:
    # ── Source tables ─────────────────────────────────────────────────────────
    IDP_TABLE        = "virtue_foundation.ghana.gold_idp_enriched"
    FACILITIES_TABLE = "virtue_foundation.ghana.gold_facilities_enriched"
    REGIONAL_TABLE   = "virtue_foundation.ghana.gold_regional_summary"

    # ── Output tables ─────────────────────────────────────────────────────────
    MDS_TABLE     = "virtue_foundation.ghana.gold_medical_desert_scores"
    OVERLAY_TABLE = "virtue_foundation.ghana.gold_facility_desert_overlay"

    # ── Output paths ──────────────────────────────────────────────────────────
    OUTPUT_VOLUME    = "/Volumes/virtue_foundation/ghana/rag"
    OUTPUT_WORKSPACE = (
        "/Workspace/Users/dasdeepayan08@gmail.com"
        "/databricks_accenture_hackathon_virtue_foundationtrack"
        "/databricks/notebooks"
    )

    MLFLOW_EXP  = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"
    SCHEMA_VER  = "v12"

    # ── Map defaults ──────────────────────────────────────────────────────────
    GHANA_CENTER_LAT = 7.9465
    GHANA_CENTER_LON = -1.0232
    MAP_ZOOM_START   = 7

    # ── WHO / SSA reference thresholds ───────────────────────────────────────
    WHO_DOCTORS_PER_10K     = 2.3   # doctors per 10,000 people
    WHO_BEDS_PER_10K        = 10.0  # beds per 10,000 people
    WHO_HOSPITALS_PER_100K  = 1.0   # hospitals per 100,000 people
    WHO_FACILITIES_PER_100K = 5.0   # total facilities per 100,000

    # ── Blended MDS component weights ────────────────────────────────────────
    # Weights verified to reproduce v12 regional desert labels correctly.
    W_BASE        = 0.40   # pre-computed per-capita density from summary
    W_SPECIALTY   = 0.25   # critical specialty gap scores
    W_INTEGRITY   = 0.20   # ghost probability + anomalies + quality risk
    W_CONFIDENCE  = 0.15   # data completeness + geo quality + readiness

    # ── MDS label thresholds (calibrated to v12 regional summary labels) ─────
    MDS_LABELS: List[Tuple[float, str]] = [
        (0.80, "Severe Desert"),
        (0.65, "Moderate Desert"),
        (0.45, "Marginal"),
        (0.00, "Adequate"),
    ]

    # ── Critical specialties for gap analysis ─────────────────────────────────
    CRITICAL_SPECIALTIES = [
        "emergencyMedicine",
        "generalSurgery",
        "gynecologyAndObstetrics",
        "internalMedicine",
        "pediatrics",
    ]


cfg = MapConfig()

for path in [cfg.OUTPUT_WORKSPACE, cfg.OUTPUT_VOLUME]:
    Path(path).mkdir(parents=True, exist_ok=True)

print(f"Schema version : {cfg.SCHEMA_VER}")
print(f"Workspace out  : {cfg.OUTPUT_WORKSPACE}")
print(f"Volume out     : {cfg.OUTPUT_VOLUME}")

# COMMAND ----------
# MAGIC %md ## 1. Utility Functions

# COMMAND ----------

def ensure_list(x) -> List[str]:
    """Convert any column value to a clean Python list of strings."""
    if x is None:
        return []
    if isinstance(x, float) and (x != x):
        return []
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x
                if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(v) for v in x.tolist()
                    if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
    except ImportError:
        pass
    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None", "NaN"):
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v) for v in parsed
                        if v is not None and str(v).strip() not in ("", "null", "None")]
            if parsed is None:
                return []
            return [str(parsed)]
        except (json.JSONDecodeError, ValueError):
            pass
        if "," in s:
            return [v.strip().strip('"').strip("'") for v in s.split(",") if v.strip()]
        return [s]
    return []


def safe_float(v, default: float = 0.0) -> float:
    try:
        if v is None:
            return default
        s = str(v).strip()
        if s in ("nan", "None", "null", "", "NaN"):
            return default
        return float(s)
    except (ValueError, TypeError):
        return default


def safe_int(v, default: int = 0) -> int:
    try:
        if v is None:
            return default
        s = str(v).strip()
        if s in ("nan", "None", "null", "", "NaN"):
            return default
        return int(float(s))
    except (ValueError, TypeError):
        return default


def safe_str(v, default: str = "") -> str:
    if v is None:
        return default
    s = str(v).strip()
    return default if s in ("None", "nan", "null", "", "NaN") else s


def safe_bool(v, default: bool = False) -> bool:
    if v is None:
        return default
    if isinstance(v, bool):
        return v
    try:
        import numpy as _np
        if isinstance(v, _np.bool_):
            return bool(v)
    except ImportError:
        pass
    s = str(v).strip().lower()
    if s in ("true", "1", "yes", "t"):
        return True
    if s in ("false", "0", "no", "f", "nan", "none", "null", ""):
        return False
    return default


def mds_label(score: float) -> str:
    """Map a 0–1 MDS score to a label using v12-calibrated thresholds."""
    for threshold, label in cfg.MDS_LABELS:
        if score >= threshold:
            return label
    return "Adequate"


def mds_color(score: float) -> str:
    """Hex colour for a single facility/region marker."""
    if score > 0.80: return "#b91c1c"   # Severe Desert   — deep red
    if score > 0.65: return "#ea580c"   # Moderate Desert — orange
    if score > 0.45: return "#ca8a04"   # Marginal        — amber
    return "#16a34a"                     # Adequate        — green


def choropleth_color(score: float) -> str:
    """Hex colour for region choropleth fill."""
    if score > 0.80: return "#991b1b"
    if score > 0.65: return "#c2410c"
    if score > 0.45: return "#b45309"
    return "#15803d"


def _clip01(v: float) -> float:
    return max(0.0, min(1.0, v))


def assert_unique_columns(cols: List[str], label: str) -> None:
    """Fail fast on exact or case-insensitive duplicate columns before Delta writes."""
    exact_seen, exact_dup = set(), []
    ci_seen, ci_dup = set(), []
    for c in cols:
        if c in exact_seen:
            exact_dup.append(c)
        exact_seen.add(c)
        key = c.lower()
        if key in ci_seen:
            ci_dup.append(c)
        ci_seen.add(key)
    if exact_dup or ci_dup:
        raise ValueError(
            f"{label} has duplicate columns. exact={exact_dup}, case_insensitive={ci_dup}"
        )


def assert_unique_pdf_columns(df: pd.DataFrame, label: str) -> pd.DataFrame:
    assert_unique_columns(list(df.columns), label)
    return df


def assert_unique_spark_columns(df, label: str):
    assert_unique_columns(df.columns, label)
    return df


# COMMAND ----------
# MAGIC %md ## 2. Load Source Data

# COMMAND ----------

# ── IDP-enriched (preferred, 190 cols) ────────────────────────────────────────
idp_pd: Optional[pd.DataFrame] = None
try:
    idp_pd = spark.table(cfg.IDP_TABLE).toPandas()
    print(f"[IDP]        {cfg.IDP_TABLE}: {len(idp_pd):,} rows, {len(idp_pd.columns)} cols")
except Exception as e:
    print(f"[IDP]        UNAVAILABLE ({e})")

# ── Facilities enriched (v12, 179 cols) ───────────────────────────────────────
try:
    fac_pd = spark.table(cfg.FACILITIES_TABLE).toPandas()
    print(f"[Facilities] {cfg.FACILITIES_TABLE}: {len(fac_pd):,} rows, {len(fac_pd.columns)} cols")
    fac_loaded = True
except Exception as e:
    print(f"[Facilities] UNAVAILABLE ({e})")
    fac_pd = pd.DataFrame()
    fac_loaded = False

# ── Regional summary (v12, 70 cols, authoritative per-capita metrics) ─────────
try:
    reg_pd = spark.table(cfg.REGIONAL_TABLE).toPandas()
    print(f"[Regional]   {cfg.REGIONAL_TABLE}: {len(reg_pd):,} rows, {len(reg_pd.columns)} cols")
    reg_loaded = True
except Exception as e:
    print(f"[Regional]   UNAVAILABLE ({e})")
    reg_pd = pd.DataFrame()
    reg_loaded = False

if not fac_loaded:
    raise RuntimeError("Facilities table is required. Cannot compute desert scores.")

# ── Merge IDP into facilities (IDP rows take priority for enriched fields) ─────
# IDP has richer quality signals; we prefer it for the 50 rows it covers.
if idp_pd is not None and len(idp_pd) > 0:
    assert_unique_pdf_columns(idp_pd, "IDP pandas input")
    assert_unique_pdf_columns(fac_pd, "facilities pandas input")
    idp_ids = set(idp_pd["unique_id"].dropna().astype(str))
    fac_non_idp = fac_pd[~fac_pd["unique_id"].astype(str).isin(idp_ids)]
    combined_pd = pd.concat([idp_pd, fac_non_idp], ignore_index=True, sort=False)
    assert_unique_pdf_columns(combined_pd, "combined facility frame")
    print(f"\n[Combined]   IDP({len(idp_pd)}) + Facilities-non-IDP({len(fac_non_idp)}) = {len(combined_pd)} rows")
else:
    combined_pd = fac_pd.copy()
    assert_unique_pdf_columns(combined_pd, "combined facility frame")
    print(f"\n[Combined]   Facilities only: {len(combined_pd)} rows (IDP not available)")

# COMMAND ----------
# MAGIC %md ## 3. Schema Normalization

# COMMAND ----------

def normalize_facility_frame(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the combined facility dataframe to a consistent internal schema.
    Handles both IDP (190 cols) and facilities (179 cols) field names.
    Coerces types; fills sensible defaults.
    """
    df = df.copy()

    # ── Numeric coercion ──────────────────────────────────────────────────────
    numeric_cols = [
        "number_doctors_int", "capacity_int", "data_completeness_score",
        "medical_desert_score", "latitude", "longitude",
        "procedure_count", "equipment_count", "capability_count", "specialty_count",
        "total_stat_anomalies",
        # v12 quality / risk signals
        "geo_quality_score", "clinical_completeness", "location_completeness",
        "contact_completeness", "ghost_probability_score", "quality_risk_score",
        "clinical_risk_score", "operational_risk_score", "integrity_risk_score",
        "emergency_readiness_score", "critical_care_score", "rag_quality_score",
        "clinical_complexity_score", "evidence_weight",
        "specialty_direct_count", "specialty_inferred_count",
        "geo_precision_tier", "quality_flag_count",
        "service_richness_score", "infrastructure_completeness_score",
        "referral_complexity_score", "healthcare_maturity_score",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    # ── Boolean coercion ──────────────────────────────────────────────────────
    bool_cols = [
        "is_hospital", "is_clinic", "is_ngo", "is_public", "is_private",
        "is_faith_based", "is_government", "is_teaching_hospital",
        "is_referral_center", "is_specialist_hospital", "is_military_hospital",
        "has_emergency_medicine", "has_surgery", "has_obstetrics", "has_pediatrics",
        "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health",
        "accepts_volunteers_bool", "capability_is_valid", "is_rag_ready",
        "is_search_ready", "is_planning_ready", "is_clinical_ready",
        "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
        "stat_anomaly_clinic_claims_icu", "stat_anomaly_ghost_facility",
        "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth",
        "geo_contradiction_flag", "geo_region_mismatch", "ngo_serves_ghana",
        "has_procedures", "has_equipment", "has_capabilities", "has_specialties",
    ]
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda v: True if str(v).strip().lower() in ("true", "1", "yes")
                else (False if str(v).strip().lower() in ("false", "0", "no", "nan", "none", "null", "")
                      else False)
            )
        else:
            df[col] = False  # safe default

    # ── Region normalisation ───────────────────────────────────────────────────
    if "region_normalised" not in df.columns:
        df["region_normalised"] = "Unknown"
    df["region_normalised"] = df["region_normalised"].fillna("Unknown").astype(str).str.strip()

    # ── City fallback ──────────────────────────────────────────────────────────
    if "city_clean" not in df.columns:
        if "address_city" in df.columns:
            df["city_clean"] = df["address_city"].fillna("")
        else:
            df["city_clean"] = ""

    # ── Default quality fields when absent (facilities table) ─────────────────
    if "geo_quality_score" not in df.columns:
        df["geo_quality_score"] = 0.5
    if "ghost_probability_score" not in df.columns:
        df["ghost_probability_score"] = 0.0
    if "emergency_readiness_score" not in df.columns:
        df["emergency_readiness_score"] = 0.0
    if "critical_care_score" not in df.columns:
        df["critical_care_score"] = 0.0
    if "rag_quality_score" not in df.columns:
        df["rag_quality_score"] = 0.5
    if "evidence_weight" not in df.columns:
        df["evidence_weight"] = 0.5
    if "clinical_completeness" not in df.columns:
        df["clinical_completeness"] = df.get("data_completeness_score", pd.Series(0.5))
    if "capability_is_valid" not in df.columns:
        df["capability_is_valid"] = True

    # ── Volunteer acceptance (accepts_volunteers_bool may be sparse) ───────────
    if "accepts_volunteers_bool" not in df.columns:
        df["accepts_volunteers_bool"] = False
    else:
        # acceptsvolunteers string column as fallback
        av_bool = df["accepts_volunteers_bool"].fillna(False)
        if "acceptsvolunteers" in df.columns:
            av_str = df["acceptsvolunteers"].apply(
                lambda v: str(v).strip().lower() in ("true", "1", "yes")
            )
            df["accepts_volunteers_bool"] = av_bool | av_str
        else:
            df["accepts_volunteers_bool"] = av_bool.astype(bool)

    return df


def normalize_regional_frame(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the regional summary dataframe.
    Coerces numeric / list-type fields; keeps authoritative per-capita metrics.
    """
    df = df.copy()

    numeric_cols = [
        "total_facilities", "clinical_facility_count", "hospital_count", "clinic_count",
        "public_facilities", "private_facilities", "ngo_count", "faith_based_count",
        "government_facilities", "teaching_hospital_count", "referral_center_count",
        "specialist_hospital_count", "volunteer_facilities",
        "avg_doctors", "total_doctors", "avg_bed_capacity", "total_beds",
        "avg_completeness", "avg_geo_quality", "avg_clinical_complexity",
        "avg_evidence_weight", "avg_ghost_probability", "avg_quality_risk",
        "avg_facility_desert_score", "avg_emergency_readiness", "avg_critical_care_score",
        "avg_service_richness_score", "avg_infrastructure_completeness_score",
        "avg_referral_complexity_score", "avg_healthcare_maturity_score",
        "emergency_medicine_facilities", "obstetrics_facilities", "surgery_facilities",
        "pediatrics_facilities", "icu_facilities", "infectious_disease_facilities",
        "radiology_facilities", "mental_health_facilities",
        "facilities_with_procedures", "facilities_with_equipment", "facilities_with_capabilities",
        "total_region_anomalies", "rag_ready_count", "rag_ready_rate", "clinical_ready_count",
        "region_population",
        "facilities_per_100k", "hospitals_per_100k", "beds_per_100k", "doctors_per_100k",
        "icu_facilities_per_100k", "surgery_facilities_per_100k", "maternity_facilities_per_100k",
        "maternity_gap_score", "emergency_gap_score", "icu_gap_score",
        "surgical_access_gap_score", "public_private_imbalance_score",
        "critical_specialty_gap_count", "medical_desert_score",
        "region_centroid_lat", "region_centroid_lon",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        else:
            df[col] = 0.0

    if "region_normalised" not in df.columns:
        df["region_normalised"] = "Unknown"
    df["region_normalised"] = df["region_normalised"].fillna("Unknown").astype(str).str.strip()

    # Centroid fallback
    df["region_centroid_lat"] = df["region_centroid_lat"].replace(0.0, cfg.GHANA_CENTER_LAT)
    df["region_centroid_lon"] = df["region_centroid_lon"].replace(0.0, cfg.GHANA_CENTER_LON)

    return df


print("Normalizing facility frame…")
combined_pd = normalize_facility_frame(combined_pd)
print(f"  Combined facilities: {len(combined_pd):,} rows, {len(combined_pd.columns)} cols")
print(f"  Regions represented: {combined_pd['region_normalised'].nunique()}")
print(f"  Lat/lon available  : {((combined_pd['latitude'] != 0) & (combined_pd['longitude'] != 0)).sum():,}")

if reg_loaded:
    print("Normalizing regional frame…")
    reg_pd = normalize_regional_frame(reg_pd)
    print(f"  Regional summary   : {len(reg_pd):,} rows, {len(reg_pd.columns)} cols")

# COMMAND ----------
# MAGIC %md ## 4. Part A — Blended Medical Desert Score (Region Level)
# MAGIC
# MAGIC **Design rationale:**
# MAGIC - `gold_regional_summary_v12` is the **primary authority** for per-capita density metrics.
# MAGIC   It carries real population estimates, facility/doctor/bed per-capita rates, and pre-computed
# MAGIC   gap scores — do not recompute from raw facility rows (too sparse, esp. doctors/beds).
# MAGIC - We compute a **blended MDS** that incorporates 4 components, each drawn from the
# MAGIC   summary table's authoritative fields plus v12 quality/integrity signals.
# MAGIC - **Desert severity is kept separate from data confidence** — the output table stores both
# MAGIC   `medical_desert_score` (severity) and `score_confidence` (data trustworthiness).

# COMMAND ----------

def _density_component(r: pd.Series) -> float:
    """
    Density component: how under-served is the region in terms of basic facility density?
    Uses per-capita metrics from regional summary (authoritative, population-normalised).
    Returns 0 (well-served) → 1 (severely under-served).
    """
    # Normalise each metric against WHO/SSA targets (1.0 = at or above target → 0 desert pressure)
    fac_score  = _clip01(safe_float(r.get("facilities_per_100k")) / cfg.WHO_FACILITIES_PER_100K)
    hosp_score = _clip01(safe_float(r.get("hospitals_per_100k")) / cfg.WHO_HOSPITALS_PER_100K)
    # Doctors and beds: convert from per-100k → per-10k for WHO comparison
    docs_score = _clip01((safe_float(r.get("doctors_per_100k")) / 10.0) / cfg.WHO_DOCTORS_PER_10K)
    beds_score = _clip01((safe_float(r.get("beds_per_100k")) / 10.0)    / cfg.WHO_BEDS_PER_10K)

    # ICU and surgery as bonus availability signals
    icu_score = _clip01(safe_float(r.get("icu_facilities_per_100k")) / 0.5)   # 0.5/100k = baseline
    surg_score= _clip01(safe_float(r.get("surgery_facilities_per_100k")) / 1.0)

    # Weighted coverage score (1 = fully covered)
    coverage = (
        fac_score  * 0.25 +
        hosp_score * 0.25 +
        docs_score * 0.20 +
        beds_score * 0.15 +
        icu_score  * 0.075 +
        surg_score * 0.075
    )
    return _clip01(1.0 - coverage)   # invert: high = desert


def _specialty_gap_component(r: pd.Series) -> float:
    """
    Specialty gap component: composite of pre-computed gap scores from regional summary.
    gap_score = 0 (fully covered) → 1 (zero coverage).
    Returns 0 (no gap) → 1 (severe gap).
    """
    em_gap   = safe_float(r.get("emergency_gap_score"),       0.0)
    icu_gap  = safe_float(r.get("icu_gap_score"),             0.0)
    surg_gap = safe_float(r.get("surgical_access_gap_score"), 0.0)
    mat_gap  = safe_float(r.get("maternity_gap_score"),       0.0)

    # Critical specialty gap count (0–5+ missing)
    gap_count = safe_int(r.get("critical_specialty_gap_count"), 0)
    gap_pct   = _clip01(gap_count / max(len(cfg.CRITICAL_SPECIALTIES), 1))

    # Weighted specialty gap (emergency and ICU most critical)
    gap_composite = (
        em_gap   * 0.35 +
        icu_gap  * 0.25 +
        surg_gap * 0.20 +
        mat_gap  * 0.10 +
        gap_pct  * 0.10
    )
    return _clip01(gap_composite)


def _integrity_component(r: pd.Series, grp_fac: Optional[pd.DataFrame]) -> float:
    """
    Integrity component: how much does data quality risk inflate the apparent capability?
    High ghost probability + high anomalies = we can trust fewer facility claims.
    Returns 0 (trustworthy) → 1 (severe integrity concern).
    """
    ghost_prob    = safe_float(r.get("avg_ghost_probability"), 0.0)
    quality_risk  = safe_float(r.get("avg_quality_risk"),      0.0)
    anomaly_count = safe_int(r.get("total_region_anomalies"),  0)
    total_fac     = max(safe_int(r.get("total_facilities"), 1), 1)

    # Normalise anomaly rate
    anomaly_rate = _clip01(anomaly_count / (total_fac * 3.0))   # 3 anomalies/facility = max

    # Facility-level integrity signals (if available from combined df)
    avg_integrity_risk = 0.0
    avg_infra_gap = 0.0
    if grp_fac is not None and len(grp_fac) > 0 and "integrity_risk_score" in grp_fac.columns:
        avg_integrity_risk = _clip01(safe_float(grp_fac["integrity_risk_score"].mean()))
    if grp_fac is not None and len(grp_fac) > 0 and "infrastructure_completeness_score" in grp_fac.columns:
        avg_infra_gap = _clip01(1.0 - safe_float(grp_fac["infrastructure_completeness_score"].mean()))

    integrity = (
        ghost_prob         * 0.35 +
        quality_risk       * 0.20 +
        anomaly_rate       * 0.18 +
        avg_integrity_risk * 0.15 +
        avg_infra_gap      * 0.12
    )
    return _clip01(integrity)


def _confidence_component(r: pd.Series, grp_fac: Optional[pd.DataFrame]) -> float:
    """
    Confidence component: how complete and trustworthy is the data?
    LOW confidence ≠ HIGH desert severity; tracked separately for planners.
    Returns 0 (high confidence) → 1 (low confidence / high uncertainty).
    """
    avg_completeness = safe_float(r.get("avg_completeness"),  0.5)
    avg_geo_quality  = safe_float(r.get("avg_geo_quality"),   0.5)
    rag_ready_rate   = safe_float(r.get("rag_ready_rate"),    50.0) / 100.0  # stored as 0–100

    # Facility-level confidence signals
    avg_rag_quality = 0.5
    avg_maturity = 0.5
    geo_contradiction_rate = 0.0
    if grp_fac is not None and len(grp_fac) > 0:
        if "rag_quality_score" in grp_fac.columns:
            avg_rag_quality = _clip01(safe_float(grp_fac["rag_quality_score"].mean()))
        if "healthcare_maturity_score" in grp_fac.columns:
            avg_maturity = _clip01(safe_float(grp_fac["healthcare_maturity_score"].mean()))
        if "geo_contradiction_flag" in grp_fac.columns:
            geo_contradiction_rate = _clip01(
                grp_fac["geo_contradiction_flag"].sum() / max(len(grp_fac), 1)
            )

    confidence_score = (
        avg_completeness      * 0.25 +
        avg_geo_quality       * 0.22 +
        rag_ready_rate        * 0.18 +
        avg_rag_quality       * 0.13 +
        avg_maturity          * 0.12 +
        (1.0 - geo_contradiction_rate) * 0.10
    )
    return _clip01(1.0 - confidence_score)   # invert: low confidence = high uncertainty


def compute_blended_mds(
    reg_df: pd.DataFrame,
    combined_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute blended Medical Desert Score for each region.

    Strategy:
    - gold_regional_summary_v12 is the **primary authority** for per-capita density metrics.
    - Facility frame adds v12 quality/risk signals per region.
    - Output stores both severity (mds) and confidence as separate fields.
    - recommended_actions and missing specialties taken directly from summary table.
    """
    records: List[Dict] = []

    for _, r in reg_df.iterrows():
        region = safe_str(r.get("region_normalised"), "Unknown")
        if not region or region in ("nan", "None"):
            continue

        # Facility group for this region (from combined frame)
        grp_mask = combined_df["region_normalised"] == region
        grp_fac  = combined_df[grp_mask] if grp_mask.any() else None

        # ── 4 scoring components ──────────────────────────────────────────────
        density_comp   = _density_component(r)
        specialty_comp = _specialty_gap_component(r)
        integrity_comp = _integrity_component(r, grp_fac)
        confidence_comp= _confidence_component(r, grp_fac)

        # ── Blended MDS ────────────────────────────────────────────────────────
        blended_mds = _clip01(
            cfg.W_BASE       * density_comp   +
            cfg.W_SPECIALTY  * specialty_comp +
            cfg.W_INTEGRITY  * integrity_comp +
            cfg.W_CONFIDENCE * confidence_comp
        )

        # ── Score confidence (separate from severity) ─────────────────────────
        avg_completeness = safe_float(r.get("avg_completeness"), 0.5)
        avg_geo_quality  = safe_float(r.get("avg_geo_quality"),  0.5)
        score_confidence = _clip01(
            0.50 * avg_completeness +
            0.30 * avg_geo_quality  +
            0.20 * (1.0 - safe_float(r.get("avg_ghost_probability"), 0.0))
        )

        # ── Authoritative reference score from regional summary ───────────────
        summary_mds = safe_float(r.get("medical_desert_score"), blended_mds)
        # Final MDS blends our computed score with the authoritative summary score
        # (summary_mds anchors us to the pipeline's calibrated population-weighted score)
        final_mds = _clip01(0.60 * summary_mds + 0.40 * blended_mds)

        label    = safe_str(r.get("desert_label"), mds_label(final_mds))
        centroid_lat = safe_float(r.get("region_centroid_lat"), cfg.GHANA_CENTER_LAT)
        centroid_lon = safe_float(r.get("region_centroid_lon"), cfg.GHANA_CENTER_LON)

        # ── Recommended actions + missing specialties (from summary table) ────
        actions      = ensure_list(r.get("recommended_actions"))
        miss_specs   = ensure_list(r.get("missing_critical_specialties"))
        all_specs    = ensure_list(r.get("all_specialties"))

        # ── Score rationale (for citations / planner transparency) ────────────
        rationale_parts: List[str] = []
        if density_comp > 0.6:
            rationale_parts.append(
                f"Density gap: {safe_float(r.get('facilities_per_100k')):.2f}/100k facilities, "
                f"{safe_float(r.get('doctors_per_100k')):.2f}/100k doctors"
            )
        if specialty_comp > 0.5:
            rationale_parts.append(f"Missing {safe_int(r.get('critical_specialty_gap_count'))} critical specialties: "
                                   f"{', '.join(miss_specs[:3])}")
        if integrity_comp > 0.4:
            rationale_parts.append(
                f"Integrity concern: ghost_prob={safe_float(r.get('avg_ghost_probability')):.2f}, "
                f"anomalies={safe_int(r.get('total_region_anomalies'))}"
            )
        if not rationale_parts:
            rationale_parts.append("No critical single-factor gap — composite score elevated across all components")
        score_rationale = "; ".join(rationale_parts)

        # ── Facility-level aggregate signals for output table ─────────────────
        n_fac     = int(grp_fac.shape[0]) if grp_fac is not None else 0
        vol_count = int(grp_fac["accepts_volunteers_bool"].sum()) if grp_fac is not None else 0
        ngo_count = int(grp_fac["is_ngo"].sum())                  if grp_fac is not None else 0
        avg_service_richness = (
            safe_float(grp_fac["service_richness_score"].mean())
            if grp_fac is not None and "service_richness_score" in grp_fac.columns else
            safe_float(r.get("avg_service_richness_score"), 0.0)
        )
        avg_infra_completeness = (
            safe_float(grp_fac["infrastructure_completeness_score"].mean())
            if grp_fac is not None and "infrastructure_completeness_score" in grp_fac.columns else
            safe_float(r.get("avg_infrastructure_completeness_score"), 0.0)
        )
        avg_referral_complexity = (
            safe_float(grp_fac["referral_complexity_score"].mean())
            if grp_fac is not None and "referral_complexity_score" in grp_fac.columns else
            safe_float(r.get("avg_referral_complexity_score"), 0.0)
        )
        avg_healthcare_maturity = (
            safe_float(grp_fac["healthcare_maturity_score"].mean())
            if grp_fac is not None and "healthcare_maturity_score" in grp_fac.columns else
            safe_float(r.get("avg_healthcare_maturity_score"), 0.0)
        )

        records.append({
            # Identity
            "region":              region,
            "schema_version":      cfg.SCHEMA_VER,
            "scored_at":           datetime.now(timezone.utc).isoformat(),
            # Summary-sourced counts
            "total_facilities":    safe_int(r.get("total_facilities")),
            "hospital_count":      safe_int(r.get("hospital_count")),
            "clinic_count":        safe_int(r.get("clinic_count")),
            "ngo_count":           safe_int(r.get("ngo_count"))       or ngo_count,
            "volunteer_facilities":safe_int(r.get("volunteer_facilities")) or vol_count,
            "teaching_hospitals":  safe_int(r.get("teaching_hospital_count")),
            "referral_centers":    safe_int(r.get("referral_center_count")),
            "public_facilities":   safe_int(r.get("public_facilities")),
            "private_facilities":  safe_int(r.get("private_facilities")),
            # Workforce
            "total_doctors":       safe_int(r.get("total_doctors")),
            "total_beds":          safe_int(r.get("total_beds")),
            "doctors_per_100k":    round(safe_float(r.get("doctors_per_100k")), 4),
            "beds_per_100k":       round(safe_float(r.get("beds_per_100k")), 4),
            "facilities_per_100k": round(safe_float(r.get("facilities_per_100k")), 4),
            "hospitals_per_100k":  round(safe_float(r.get("hospitals_per_100k")), 4),
            "region_population":   safe_int(r.get("region_population")),
            # Specialty coverage
            "emergency_medicine_facilities": safe_int(r.get("emergency_medicine_facilities")),
            "surgery_facilities":            safe_int(r.get("surgery_facilities")),
            "obstetrics_facilities":         safe_int(r.get("obstetrics_facilities")),
            "icu_facilities":                safe_int(r.get("icu_facilities")),
            "pediatrics_facilities":         safe_int(r.get("pediatrics_facilities")),
            "infectious_disease_facilities": safe_int(r.get("infectious_disease_facilities")),
            "radiology_facilities":          safe_int(r.get("radiology_facilities")),
            "mental_health_facilities":      safe_int(r.get("mental_health_facilities")),
            "critical_specialty_gap_count":  safe_int(r.get("critical_specialty_gap_count")),
            "missing_critical_specialties":  json.dumps(miss_specs),
            "all_specialties":               json.dumps(all_specs[:15]),
            # Gap scores (0=no gap, 1=complete gap)
            "emergency_gap_score":       round(safe_float(r.get("emergency_gap_score")),    4),
            "icu_gap_score":             round(safe_float(r.get("icu_gap_score")),          4),
            "surgical_access_gap_score": round(safe_float(r.get("surgical_access_gap_score")), 4),
            "maternity_gap_score":       round(safe_float(r.get("maternity_gap_score")),    4),
            # Quality
            "avg_completeness":      round(safe_float(r.get("avg_completeness")),      4),
            "avg_geo_quality":       round(safe_float(r.get("avg_geo_quality")),       4),
            "avg_ghost_probability": round(safe_float(r.get("avg_ghost_probability")), 4),
            "avg_quality_risk":      round(safe_float(r.get("avg_quality_risk")),      4),
            "total_region_anomalies":safe_int(r.get("total_region_anomalies")),
            "rag_ready_count":       safe_int(r.get("rag_ready_count")),
            "rag_ready_rate":        round(safe_float(r.get("rag_ready_rate")),        2),
            # Readiness
            "avg_emergency_readiness": round(safe_float(r.get("avg_emergency_readiness")), 4),
            "avg_critical_care_score": round(safe_float(r.get("avg_critical_care_score")), 4),
            "avg_service_richness_score": round(avg_service_richness, 4),
            "avg_infrastructure_completeness_score": round(avg_infra_completeness, 4),
            "avg_referral_complexity_score": round(avg_referral_complexity, 4),
            "avg_healthcare_maturity_score": round(avg_healthcare_maturity, 4),
            # MDS components
            "density_component":    round(density_comp,   4),
            "specialty_component":  round(specialty_comp, 4),
            "integrity_component":  round(integrity_comp, 4),
            "confidence_component": round(confidence_comp,4),
            # Scores
            "summary_mds":          round(summary_mds,  4),
            "blended_mds":          round(blended_mds,  4),
            "medical_desert_score": round(final_mds,    4),
            "desert_label":         label,
            "mds_label":            label,
            "score_confidence":     round(score_confidence, 4),
            "score_rationale":      score_rationale,
            # Geography
            "centroid_lat":         round(centroid_lat, 6),
            "centroid_lon":         round(centroid_lon, 6),
            # Planning
            "recommended_actions":  json.dumps(actions),
            "method_version":       f"blended_mds_{cfg.SCHEMA_VER}",
        })

    mds_out = (
        pd.DataFrame(records)
        .sort_values("medical_desert_score", ascending=False)
        .reset_index(drop=True)
    )
    return mds_out


# ── Run if regional summary is available; fallback to facility-only scoring ───
if reg_loaded and len(reg_pd) > 0:
    print("Computing blended MDS using gold_regional_summary_v12 as primary source…")
    mds_pd = compute_blended_mds(reg_pd, combined_pd)
    print(f"Scored {len(mds_pd)} regions")
else:
    print("Regional summary unavailable — falling back to facility-only density scoring")
    # Minimal facility-only fallback (no per-capita metrics)
    mds_records = []
    for region, grp in combined_pd.groupby("region_normalised", dropna=False):
        if not str(region) or str(region) in ("nan","None",""): continue
        n = len(grp)
        mds_score = _clip01(1.0 - safe_float(grp["data_completeness_score"].mean(), 0.5))
        mds_records.append({
            "region": str(region), "total_facilities": n,
            "medical_desert_score": round(mds_score, 4),
            "desert_label": mds_label(mds_score),
            "mds_label": mds_label(mds_score),
            "score_confidence": 0.3, "score_rationale": "Facility-only fallback (no regional summary)",
            "centroid_lat": cfg.GHANA_CENTER_LAT, "centroid_lon": cfg.GHANA_CENTER_LON,
            "recommended_actions": json.dumps(["Obtain regional summary data for full analysis"]),
        })
    mds_pd = pd.DataFrame(mds_records).sort_values("medical_desert_score", ascending=False).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"MEDICAL DESERT SCORES — {len(mds_pd)} REGIONS")
print(f"{'='*70}")
print(f"{'Region':<22}  {'MDS':>6}  {'Conf':>5}  {'Label':<18}  {'EmGap':>6}  {'ICUGap':>6}")
print("-" * 70)
for _, r in mds_pd.iterrows():
    em  = r.get("emergency_gap_score", 0.0)
    icu = r.get("icu_gap_score", 0.0)
    print(f"{r['region']:<22}  {r['medical_desert_score']:>6.3f}  "
          f"{r.get('score_confidence', 0):>5.2f}  {r['desert_label']:<18}  "
          f"{em:>6.2f}  {icu:>6.2f}")

dist = Counter(mds_pd["desert_label"])
print(f"\nDistribution: {dict(dist)}")
print(f"Avg MDS      : {mds_pd['medical_desert_score'].mean():.3f}")
print(f"Avg Confidence: {mds_pd.get('score_confidence', pd.Series([0])).mean():.3f}")

# COMMAND ----------
# MAGIC %md ## 5. Facility-Level Desert Overlay Score

# COMMAND ----------

def compute_facility_desert_overlay(
    combined_df: pd.DataFrame,
    region_mds_map: Dict[str, float],
    region_label_map: Dict[str, str],
) -> pd.DataFrame:
    """
    Compute a facility-level desert exposure score for every facility.

    Blends:
    - Regional MDS (the region-level desert severity inherited by the facility)
    - Facility-level service gap (missing critical capabilities relative to facility type)
    - Facility integrity risk (ghost probability, anomalies, low quality)
    - Facility access potential (readiness scores, volunteers, NGO status)

    Returns a dataframe with unique_id + overlay scores for persistence.
    """
    results: List[Dict] = []

    for _, row in combined_df.iterrows():
        uid    = safe_str(row.get("unique_id", ""))
        region = safe_str(row.get("region_normalised", "Unknown"))
        region_mds = safe_float(region_mds_map.get(region), 0.5)

        # ── Service gap component ─────────────────────────────────────────────
        # Count missing critical capabilities for a hospital-tier facility
        is_hosp = safe_bool(row.get("is_hospital"))
        miss_caps = 0
        cap_flags = [
            "has_emergency_medicine", "has_surgery", "has_obstetrics",
            "has_icu", "has_pediatrics",
        ]
        for flag in cap_flags:
            if not safe_bool(row.get(flag)):
                miss_caps += 1
        service_gap = _clip01(miss_caps / len(cap_flags))

        # ── Facility integrity risk ────────────────────────────────────────────
        ghost_prob     = _clip01(safe_float(row.get("ghost_probability_score"), 0.0))
        quality_risk   = _clip01(safe_float(row.get("quality_risk_score"),      0.0))
        integrity_risk = _clip01(safe_float(row.get("integrity_risk_score"),    0.0))
        anomaly_count  = safe_int(row.get("total_stat_anomalies", 0))
        cap_valid      = safe_bool(row.get("capability_is_valid", True))

        facility_integrity = _clip01(
            ghost_prob      * 0.40 +
            quality_risk    * 0.25 +
            integrity_risk  * 0.20 +
            (anomaly_count / 3.0) * 0.10 +
            (0.0 if cap_valid else 1.0) * 0.05
        )

        # ── Access potential (positive signal — reduces desert exposure) ───────
        em_ready    = _clip01(safe_float(row.get("emergency_readiness_score"), 0.0))
        crit_care   = _clip01(safe_float(row.get("critical_care_score"),       0.0))
        rag_quality = _clip01(safe_float(row.get("rag_quality_score"),         0.5))
        accepts_vol = 1.0 if safe_bool(row.get("accepts_volunteers_bool")) else 0.0
        is_ngo      = 1.0 if safe_bool(row.get("is_ngo")) else 0.0

        access_bonus = _clip01(
            em_ready    * 0.40 +
            crit_care   * 0.30 +
            rag_quality * 0.15 +
            accepts_vol * 0.10 +
            is_ngo      * 0.05
        )

        # ── Facility desert overlay ────────────────────────────────────────────
        # region MDS is the base; local facility signals modulate it
        facility_overlay = _clip01(
            0.50 * region_mds       +
            0.25 * service_gap      +
            0.15 * facility_integrity +
            0.10 * (1.0 - access_bonus)
        )

        results.append({
            "unique_id":               uid,
            "name":                    safe_str(row.get("name")),
            "region":                  region,
            "city":                    safe_str(row.get("city_clean")),
            "facility_type":           safe_str(row.get("facility_type_clean")),
            "facility_tier_label":     safe_str(row.get("facility_tier_label")),
            "latitude":                safe_float(row.get("latitude"),  cfg.GHANA_CENTER_LAT),
            "longitude":               safe_float(row.get("longitude"), cfg.GHANA_CENTER_LON),
            "region_mds":              round(region_mds,           4),
            "region_desert_label":     region_label_map.get(region, "Unknown"),
            "service_gap_score":       round(service_gap,          4),
            "facility_integrity_risk": round(facility_integrity,   4),
            "access_bonus":            round(access_bonus,          4),
            "facility_desert_overlay": round(facility_overlay,     4),
            "facility_desert_label":   mds_label(facility_overlay),
            # Capability flags
            "has_emergency_medicine":  safe_bool(row.get("has_emergency_medicine")),
            "has_surgery":             safe_bool(row.get("has_surgery")),
            "has_obstetrics":          safe_bool(row.get("has_obstetrics")),
            "has_icu":                 safe_bool(row.get("has_icu")),
            "has_radiology":           safe_bool(row.get("has_radiology")),
            "has_infectious_disease":  safe_bool(row.get("has_infectious_disease")),
            "has_mental_health":       safe_bool(row.get("has_mental_health")),
            "has_pediatrics":          safe_bool(row.get("has_pediatrics")),
            # Quality signals
            "ghost_probability_score": round(safe_float(row.get("ghost_probability_score")), 4),
            "total_stat_anomalies":    safe_int(row.get("total_stat_anomalies")),
            "capability_is_valid":     safe_bool(row.get("capability_is_valid", True)),
            "data_completeness_score": round(safe_float(row.get("data_completeness_score")), 4),
            "rag_quality_score":       round(safe_float(row.get("rag_quality_score")), 4),
            "emergency_readiness_score": round(safe_float(row.get("emergency_readiness_score")), 4),
            "critical_care_score":     round(safe_float(row.get("critical_care_score")), 4),
            "accepts_volunteers":      safe_bool(row.get("accepts_volunteers_bool")),
            "is_ngo":                  safe_bool(row.get("is_ngo")),
            "is_hospital":             safe_bool(row.get("is_hospital")),
            "is_clinic":               safe_bool(row.get("is_clinic")),
            "doctors":                 safe_int(row.get("number_doctors_int")),
            "beds":                    safe_int(row.get("capacity_int")),
            "schema_version":          cfg.SCHEMA_VER,
        })

    overlay_df = (
        pd.DataFrame(results)
        .sort_values("facility_desert_overlay", ascending=False)
        .reset_index(drop=True)
    )
    return overlay_df


# Build lookup maps
region_mds_map   = dict(zip(mds_pd["region"], mds_pd["medical_desert_score"]))
region_label_map = dict(zip(mds_pd["region"], mds_pd["desert_label"]))

print("Computing facility-level desert overlay scores…")
overlay_pd = compute_facility_desert_overlay(combined_pd, region_mds_map, region_label_map)
print(f"Facility overlay computed: {len(overlay_pd):,} rows")
print(f"Desert label distribution (facility): {Counter(overlay_pd['facility_desert_label'])}")
print(f"\nTop 5 highest facility desert overlay:")
for _, r in overlay_pd.head(5).iterrows():
    print(f"  {r['name']!r:<40} {r['region']:<20} overlay={r['facility_desert_overlay']:.3f} [{r['facility_desert_label']}]")

# COMMAND ----------
# MAGIC %md ## 6. Persist to Delta

# COMMAND ----------

# ── Helper: safe nullable Spark row ───────────────────────────────────────────
def _clean_record(row: Dict) -> Dict:
    """Convert numpy types and NaN to Python primitives for Spark."""
    out = {}
    for k, v in row.items():
        if v is None:
            out[k] = None
        elif isinstance(v, bool):
            out[k] = v
        else:
            try:
                import numpy as _np
                if isinstance(v, _np.bool_):
                    out[k] = bool(v)
                elif isinstance(v, _np.integer):
                    out[k] = int(v)
                elif isinstance(v, _np.floating):
                    out[k] = None if (v != v) else float(v)
                elif isinstance(v, float) and (v != v):
                    out[k] = None
                else:
                    out[k] = v
            except ImportError:
                if isinstance(v, float) and (v != v):
                    out[k] = None
                else:
                    out[k] = v
    return out


def _ensure_delta_column(table_name: str, column_name: str, ddl_type: str) -> None:
    existing = {field.name.lower() for field in spark.table(table_name).schema.fields}
    if column_name.lower() not in existing:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({column_name} {ddl_type})")
        print(f"   Added missing column {table_name}.{column_name} {ddl_type}")


def _merge_mds_into_table(table_name: str) -> None:
    """
    Back-fill regional MDS into an existing Delta table without rewriting the
    table from itself. This avoids DELTA_FAILED_TO_MERGE_FIELDS on score columns.
    """
    _ensure_delta_column(table_name, "medical_desert_score", "DOUBLE")
    _ensure_delta_column(table_name, "desert_label", "STRING")
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING (
            SELECT
                CAST(region AS STRING) AS region_normalised,
                CAST(medical_desert_score AS DOUBLE) AS medical_desert_score,
                CAST(desert_label AS STRING) AS desert_label
            FROM {cfg.MDS_TABLE}
        ) AS source
        ON target.region_normalised = source.region_normalised
        WHEN MATCHED THEN UPDATE SET
            target.medical_desert_score = COALESCE(source.medical_desert_score, target.medical_desert_score),
            target.desert_label = COALESCE(source.desert_label, target.desert_label)
    """)


# ── 6a. Write region MDS table ────────────────────────────────────────────────
mds_schema = T.StructType([
    T.StructField("region",                 T.StringType(), True),
    T.StructField("schema_version",         T.StringType(), True),
    T.StructField("scored_at",              T.StringType(), True),
    T.StructField("total_facilities",       T.IntegerType(), True),
    T.StructField("hospital_count",         T.IntegerType(), True),
    T.StructField("clinic_count",           T.IntegerType(), True),
    T.StructField("ngo_count",              T.IntegerType(), True),
    T.StructField("volunteer_facilities",   T.IntegerType(), True),
    T.StructField("teaching_hospitals",     T.IntegerType(), True),
    T.StructField("referral_centers",       T.IntegerType(), True),
    T.StructField("public_facilities",      T.IntegerType(), True),
    T.StructField("private_facilities",     T.IntegerType(), True),
    T.StructField("total_doctors",          T.IntegerType(), True),
    T.StructField("total_beds",             T.IntegerType(), True),
    T.StructField("doctors_per_100k",       T.DoubleType(), True),
    T.StructField("beds_per_100k",          T.DoubleType(), True),
    T.StructField("facilities_per_100k",    T.DoubleType(), True),
    T.StructField("hospitals_per_100k",     T.DoubleType(), True),
    T.StructField("region_population",      T.IntegerType(), True),
    T.StructField("emergency_medicine_facilities", T.IntegerType(), True),
    T.StructField("surgery_facilities",     T.IntegerType(), True),
    T.StructField("obstetrics_facilities",  T.IntegerType(), True),
    T.StructField("icu_facilities",         T.IntegerType(), True),
    T.StructField("pediatrics_facilities",  T.IntegerType(), True),
    T.StructField("infectious_disease_facilities", T.IntegerType(), True),
    T.StructField("radiology_facilities",   T.IntegerType(), True),
    T.StructField("mental_health_facilities", T.IntegerType(), True),
    T.StructField("critical_specialty_gap_count", T.IntegerType(), True),
    T.StructField("missing_critical_specialties", T.StringType(), True),
    T.StructField("all_specialties",        T.StringType(), True),
    T.StructField("emergency_gap_score",    T.DoubleType(), True),
    T.StructField("icu_gap_score",          T.DoubleType(), True),
    T.StructField("surgical_access_gap_score", T.DoubleType(), True),
    T.StructField("maternity_gap_score",    T.DoubleType(), True),
    T.StructField("avg_completeness",       T.DoubleType(), True),
    T.StructField("avg_geo_quality",        T.DoubleType(), True),
    T.StructField("avg_ghost_probability",  T.DoubleType(), True),
    T.StructField("avg_quality_risk",       T.DoubleType(), True),
    T.StructField("total_region_anomalies", T.IntegerType(), True),
    T.StructField("rag_ready_count",        T.IntegerType(), True),
    T.StructField("rag_ready_rate",         T.DoubleType(), True),
    T.StructField("avg_emergency_readiness", T.DoubleType(), True),
    T.StructField("avg_critical_care_score", T.DoubleType(), True),
    T.StructField("avg_service_richness_score", T.DoubleType(), True),
    T.StructField("avg_infrastructure_completeness_score", T.DoubleType(), True),
    T.StructField("avg_referral_complexity_score", T.DoubleType(), True),
    T.StructField("avg_healthcare_maturity_score", T.DoubleType(), True),
    T.StructField("density_component",      T.DoubleType(), True),
    T.StructField("specialty_component",    T.DoubleType(), True),
    T.StructField("integrity_component",    T.DoubleType(), True),
    T.StructField("confidence_component",   T.DoubleType(), True),
    T.StructField("summary_mds",            T.DoubleType(), True),
    T.StructField("blended_mds",            T.DoubleType(), True),
    T.StructField("medical_desert_score",   T.DoubleType(), True),
    T.StructField("desert_label",           T.StringType(), True),
    T.StructField("mds_label",              T.StringType(), True),
    T.StructField("score_confidence",       T.DoubleType(), True),
    T.StructField("score_rationale",        T.StringType(), True),
    T.StructField("centroid_lat",           T.DoubleType(), True),
    T.StructField("centroid_lon",           T.DoubleType(), True),
    T.StructField("recommended_actions",    T.StringType(), True),
    T.StructField("method_version",         T.StringType(), True),
])

try:
    mds_spark = spark.createDataFrame(
        [_clean_record(r) for r in mds_pd.to_dict("records")],
        schema=mds_schema,
    )
    mds_spark = assert_unique_spark_columns(mds_spark, "region MDS Spark output")
    (mds_spark.write.format("delta")
     .option("overwriteSchema", "true")
     .mode("overwrite")
     .saveAsTable(cfg.MDS_TABLE))
    print(f"✅  Region MDS written to {cfg.MDS_TABLE}")
except Exception as e:
    print(f"❌  Region MDS write failed: {e}")

# ── 6b. Write facility overlay table ─────────────────────────────────────────
try:
    overlay_spark = spark.createDataFrame(
        [_clean_record(r) for r in overlay_pd.to_dict("records")]
    )
    overlay_spark = assert_unique_spark_columns(overlay_spark, "facility desert overlay Spark output")
    (overlay_spark.write.format("delta")
     .option("overwriteSchema", "true")
     .mode("overwrite")
     .saveAsTable(cfg.OVERLAY_TABLE))
    print(f"✅  Facility overlay written to {cfg.OVERLAY_TABLE}")
except Exception as e:
    print(f"❌  Facility overlay write failed: {e}")

# ── 6c. Back-fill MDS into gold_idp_enriched ─────────────────────────────────
try:
    _merge_mds_into_table(cfg.IDP_TABLE)
    print(f"✅  MDS back-filled to {cfg.IDP_TABLE}")
except Exception as e:
    print(f"   IDP back-fill skipped: {e}")

# ── 6d. Back-fill MDS into gold_facilities_enriched ──────────────────────────
try:
    _merge_mds_into_table(cfg.FACILITIES_TABLE)
    print(f"✅  MDS back-filled to {cfg.FACILITIES_TABLE}")
except Exception as e:
    print(f"   Facilities back-fill skipped: {e}")

# COMMAND ----------
# MAGIC %md ## 7. Part B — Folium Map (Choropleth + Facility Cluster)

# COMMAND ----------

def _build_facility_popup(row: pd.Series, region_mds: float, region_label_str: str) -> str:
    """
    Citation-aware HTML popup for each facility marker.
    Uses v12 quality signals, enriched clinical arrays, IDP citations.
    """
    name     = safe_str(row.get("name", "Unknown"))
    ftype    = safe_str(row.get("facility_type_clean", "?")).upper()
    tier     = safe_str(row.get("facility_tier_label", ""))
    city     = safe_str(row.get("city_clean", ""))
    region   = safe_str(row.get("region_normalised", ""))
    doctors  = safe_int(row.get("number_doctors_int"))
    beds     = safe_int(row.get("capacity_int"))
    completeness    = safe_float(row.get("data_completeness_score"))
    rag_quality     = safe_float(row.get("rag_quality_score"))
    ghost_prob      = safe_float(row.get("ghost_probability_score"))
    em_readiness    = safe_float(row.get("emergency_readiness_score"))
    crit_care       = safe_float(row.get("critical_care_score"))
    cap_valid       = safe_bool(row.get("capability_is_valid", True))
    has_anomaly     = not cap_valid or safe_int(row.get("total_stat_anomalies")) > 0
    accepts_vol     = safe_bool(row.get("accepts_volunteers_bool"))
    is_ngo          = safe_bool(row.get("is_ngo"))

    # Clinical arrays — prefer enriched (IDP); fall back to parsed
    specs  = ensure_list(row.get("specialties_enriched")) or ensure_list(row.get("specialties_parsed"))
    procs  = ensure_list(row.get("procedure_enriched"))   or ensure_list(row.get("procedure_parsed"))
    equips = ensure_list(row.get("equipment_enriched"))   or ensure_list(row.get("equipment_parsed"))
    caps   = ensure_list(row.get("capability_enriched"))  or ensure_list(row.get("capability_parsed"))

    # Citations (IDP first, then structured citations from fac)
    idp_cits  = ensure_list(row.get("idp_citations"))
    fac_cits  = ensure_list(row.get("citations"))
    cit_texts = [c for c in idp_cits if len(c) > 15]
    if not cit_texts:
        # Parse structured fac citations
        for c in fac_cits[:3]:
            try:
                obj = json.loads(c) if isinstance(c, str) else c
                if isinstance(obj, dict):
                    snip = obj.get("snippet", "")
                    if snip and len(snip) > 10:
                        cit_texts.append(f"[{obj.get('field','')}] {snip[:100]}")
            except Exception:
                if len(c) > 10:
                    cit_texts.append(c[:100])

    # Facility-level overlay score (if computed)
    overlay_score = safe_float(row.get("facility_desert_overlay"), region_mds)
    display_mds   = overlay_score if overlay_score != region_mds else region_mds
    mds_color_str = mds_color(display_mds)

    def _cap_badge(flag_val, label: str) -> str:
        ok = safe_bool(flag_val)
        c  = "#16a34a" if ok else "#dc2626"
        return f'<span style="background:{c};color:#fff;padding:1px 5px;border-radius:3px;font-size:10px;margin:1px">{label}</span>'

    caps_html = " ".join([
        _cap_badge(row.get("has_emergency_medicine"), "ER"),
        _cap_badge(row.get("has_surgery"),            "Surgery"),
        _cap_badge(row.get("has_icu"),                "ICU"),
        _cap_badge(row.get("has_obstetrics"),         "OB/GYN"),
        _cap_badge(row.get("has_radiology"),          "Radiology"),
        _cap_badge(row.get("has_infectious_disease"), "Infect."),
        _cap_badge(row.get("has_mental_health"),      "Mental"),
        _cap_badge(row.get("has_pediatrics"),         "Pediatrics"),
    ])

    anom_badge = (
        '<span style="background:#dc2626;color:#fff;padding:2px 6px;'
        'border-radius:3px;font-size:10px;margin-left:4px">⚠ ANOMALY</span>'
        if has_anomaly else ""
    )
    vol_badge = (
        '<span style="background:#7c3aed;color:#fff;padding:2px 6px;'
        'border-radius:3px;font-size:10px;margin:2px 0">🙋 Volunteers Welcome</span>'
        if accepts_vol else ""
    )
    ngo_badge = (
        '<span style="background:#0891b2;color:#fff;padding:2px 6px;'
        'border-radius:3px;font-size:10px;margin:2px 0">🌐 NGO</span>'
        if is_ngo else ""
    )

    # Citation block
    cit_html = ""
    if cit_texts:
        items = "".join(
            f'<li style="font-size:10px;color:#6b7280">{c[:110]}</li>'
            for c in cit_texts[:3]
        )
        cit_html = f'<b>Citations:</b><ul style="margin:2px 0 0 12px;padding:0">{items}</ul>'

    # Quality bar
    def _bar(val, label, color="#1d4ed8"):
        pct = max(0, min(100, int(val * 100)))
        return (f'<div style="font-size:10px;margin:1px 0">{label}: '
                f'<div style="display:inline-block;width:80px;background:#e5e7eb;border-radius:2px;vertical-align:middle">'
                f'<div style="width:{pct}%;height:6px;background:{color};border-radius:2px"></div></div>'
                f' {val:.2f}</div>')

    return f"""
    <div style="font-family:sans-serif;width:360px;max-height:520px;overflow:auto;font-size:12px">
        <h4 style="margin:0 0 4px">{name}{anom_badge}</h4>
        <p style="margin:2px 0;color:#374151">
            <b>{ftype}</b>{(' — '+tier) if tier else ''} &nbsp;|&nbsp; {city}, {region}
        </p>
        <hr style="margin:5px 0;border:0;border-top:1px solid #e5e7eb">
        <div style="background:{mds_color_str};color:#fff;padding:3px 8px;border-radius:4px;
                    font-size:11px;display:inline-block;margin-bottom:3px">
            🏜 {region_label_str} (MDS={display_mds:.3f})
        </div>
        {'<br>' + vol_badge + ' ' + ngo_badge if (accepts_vol or is_ngo) else ''}
        <br><br>
        <div>{caps_html}</div>
        <br>
        {'<p style="margin:2px 0"><b>Specialties:</b> ' + ', '.join(specs[:5]) + '</p>' if specs else ''}
        {'<p style="margin:2px 0"><b>Procedures:</b> ' + '; '.join(procs[:3]) + '</p>' if procs else ''}
        {'<p style="margin:2px 0"><b>Equipment:</b> ' + '; '.join(equips[:3]) + '</p>' if equips else ''}
        {'<p style="margin:2px 0"><b>Capabilities:</b> ' + '; '.join(caps[:3]) + '</p>' if caps else ''}
        <p style="margin:4px 0;color:#6b7280">Doctors: {doctors} | Beds: {beds}</p>
        <hr style="margin:4px 0;border:0;border-top:1px solid #e5e7eb">
        {_bar(completeness,    "Completeness",    "#1d4ed8")}
        {_bar(rag_quality,     "RAG Quality",     "#7c3aed")}
        {_bar(em_readiness,    "EM Readiness",    "#15803d")}
        {_bar(crit_care,       "Critical Care",   "#b91c1c")}
        {_bar(ghost_prob,      "Ghost Risk",      "#dc2626") if ghost_prob > 0.1 else ''}
        {cit_html}
        <p style="font-size:10px;margin:3px 0;color:#9ca3af">
            ID: {safe_str(row.get('unique_id','?'))[:32]}
        </p>
    </div>
    """


def build_medical_desert_map(
    fac_df: pd.DataFrame,
    mds_df: pd.DataFrame,
    overlay_df: pd.DataFrame,
) -> folium.Map:
    """
    Layered Folium map:
    Layer 1: Region choropleth (circle markers at centroids, sized by MDS)
    Layer 2: Facility marker cluster (coloured by type/anomaly status)
    Layer 3: Facility density heat map
    Layer 4: Critical specialty gap annotations
    """
    # Build lookup maps
    region_mds_map   = dict(zip(mds_df["region"], mds_df["medical_desert_score"]))
    region_label_map = dict(zip(mds_df["region"], mds_df["desert_label"]))

    # Merge overlay scores into facility frame for richer popups
    if "unique_id" in overlay_df.columns and "facility_desert_overlay" in overlay_df.columns:
        overlay_slim = overlay_df[["unique_id", "facility_desert_overlay", "service_gap_score",
                                   "facility_integrity_risk"]].copy()
        fac_df = fac_df.merge(overlay_slim, on="unique_id", how="left")

    m = folium.Map(
        location   = [cfg.GHANA_CENTER_LAT, cfg.GHANA_CENTER_LON],
        zoom_start = cfg.MAP_ZOOM_START,
        tiles      = "CartoDB positron",
    )

    # ── Layer 1: Region circles (choropleth proxy) ────────────────────────────
    for _, r in mds_df.iterrows():
        mds_s = safe_float(r.get("medical_desert_score"), 0.5)
        lat   = safe_float(r.get("centroid_lat"), cfg.GHANA_CENTER_LAT)
        lon   = safe_float(r.get("centroid_lon"), cfg.GHANA_CENTER_LON)
        color = choropleth_color(mds_s)
        label = safe_str(r.get("desert_label"), "Unknown")

        miss_specs = ensure_list(r.get("missing_critical_specialties"))
        rec_acts   = ensure_list(r.get("recommended_actions"))
        em_gap     = safe_float(r.get("emergency_gap_score"))
        icu_gap    = safe_float(r.get("icu_gap_score"))
        surg_gap   = safe_float(r.get("surgical_access_gap_score"))
        confidence = safe_float(r.get("score_confidence"))

        # Top 3 actions (truncated for popup)
        act_html = "".join(
            f'<li style="font-size:11px">{a[:120]}</li>'
            for a in rec_acts[:4]
        )

        popup_html = f"""
        <div style="font-family:sans-serif;width:320px;font-size:12px">
            <h4 style="color:{color};margin:0 0 4px">{r['region']}</h4>
            <div style="background:{color};color:#fff;padding:3px 8px;border-radius:4px;
                        font-size:12px;display:inline-block;margin-bottom:6px">
                MDS = {mds_s:.3f} — {label}
            </div>
            <p style="margin:2px 0;font-size:10px;color:#6b7280">
                Score confidence: {confidence:.2f} &nbsp;|&nbsp;
                Population: {safe_int(r.get('region_population')):,}
            </p>
            <hr style="margin:6px 0;border:0;border-top:1px solid #e5e7eb">
            <b>Infrastructure:</b> {safe_int(r.get('total_facilities'))} facilities,
            {safe_int(r.get('hospital_count'))} hospitals,
            {safe_int(r.get('total_doctors'))} doctors,
            {safe_int(r.get('total_beds'))} beds<br>
            <b>Per-capita:</b> {safe_float(r.get('doctors_per_100k')):.2f} doctors/100k,
            {safe_float(r.get('beds_per_100k')):.2f} beds/100k<br>
            <hr style="margin:6px 0;border:0;border-top:1px solid #e5e7eb">
            <b>Gap scores (1=complete gap):</b><br>
            &nbsp;Emergency={em_gap:.2f} &nbsp;ICU={icu_gap:.2f} &nbsp;Surgery={surg_gap:.2f}<br>
            {'<b>Missing specialties:</b> ' + ', '.join(miss_specs[:4]) + '<br>' if miss_specs else ''}
            <b>Score rationale:</b> {safe_str(r.get('score_rationale',''))[:160]}<br>
            <hr style="margin:6px 0;border:0;border-top:1px solid #e5e7eb">
            <b>Recommended actions:</b><ul style="margin:2px 0;padding:0 0 0 14px">
            {act_html}
            </ul>
            <p style="font-size:10px;color:#6b7280;margin:4px 0">
                Anomalies: {safe_int(r.get('total_region_anomalies'))} &nbsp;|&nbsp;
                NGOs: {safe_int(r.get('ngo_count'))} &nbsp;|&nbsp;
                Volunteers: {safe_int(r.get('volunteer_facilities'))} facilities
            </p>
        </div>
        """
        folium.CircleMarker(
            location     = [lat, lon],
            radius       = max(14, 15 + mds_s * 28),
            color        = color,
            fill         = True,
            fill_color   = color,
            fill_opacity = 0.22,
            weight       = 2.5,
            popup        = folium.Popup(popup_html, max_width=340),
            tooltip      = f"📍 {r['region']} | MDS={mds_s:.3f} | {label} | Conf={confidence:.2f}",
        ).add_to(m)

    # ── Layer 2: Facility marker cluster ──────────────────────────────────────
    cluster = MarkerCluster(name="Healthcare Facilities", show=True)

    TYPE_ICON: Dict[str, Tuple[str, str]] = {
        "hospital":  ("hospital-o",  "cadetblue"),
        "clinic":    ("plus-square", "green"),
        "ngo":       ("heart",       "purple"),
        "pharmacy":  ("medkit",      "orange"),
        "dentist":   ("user-md",     "darkred"),
        "maternity": ("female",      "pink"),
    }
    DEFAULT_ICON = ("info-sign", "gray")

    geo_fac = fac_df[
        (fac_df["latitude"]  != 0) & fac_df["latitude"].notna() &
        (fac_df["longitude"] != 0) & fac_df["longitude"].notna()
    ].copy()

    for _, row in geo_fac.iterrows():
        lat    = safe_float(row.get("latitude"),  0.0)
        lon    = safe_float(row.get("longitude"), 0.0)
        if lat == 0.0 and lon == 0.0:
            continue

        ftype      = safe_str(row.get("facility_type_clean", "unknown")).lower()
        region     = safe_str(row.get("region_normalised", "Unknown"))
        has_anom   = (not safe_bool(row.get("capability_is_valid", True))
                      or safe_int(row.get("total_stat_anomalies")) > 0)
        ghost_high = safe_float(row.get("ghost_probability_score")) > 0.5
        mds_s      = safe_float(region_mds_map.get(region), 0.5)
        d_label    = region_label_map.get(region, safe_str(row.get("desert_label"), "Unknown"))

        if has_anom or ghost_high:
            icon_conf = ("warning-sign", "red")
        elif ftype in TYPE_ICON:
            icon_conf = TYPE_ICON[ftype]
        else:
            icon_conf = DEFAULT_ICON

        popup_html = _build_facility_popup(row, mds_s, d_label)
        tooltip    = (f"{safe_str(row.get('name','?'))} | "
                      f"{safe_str(row.get('facility_tier_label','?'))} | "
                      f"{safe_str(row.get('city_clean','?'))}")

        folium.Marker(
            location = [lat, lon],
            icon     = folium.Icon(icon=icon_conf[0], prefix="fa", color=icon_conf[1]),
            popup    = folium.Popup(popup_html, max_width=390),
            tooltip  = tooltip,
        ).add_to(cluster)

    cluster.add_to(m)

    # ── Layer 3: Facility density heat map ────────────────────────────────────
    heat_data = [
        [safe_float(row["latitude"]), safe_float(row["longitude"]), 1.0]
        for _, row in geo_fac.iterrows()
    ]
    if heat_data:
        HeatMap(
            heat_data, name="Facility Density Heatmap",
            radius=16, blur=14, min_opacity=0.2, show=False,
        ).add_to(m)

    # ── Layer 4: Critical specialty gap annotations ───────────────────────────
    gap_group = folium.FeatureGroup(name="Critical Specialty Gaps", show=False)
    for _, r in mds_df.iterrows():
        miss = ensure_list(r.get("missing_critical_specialties"))
        if not miss:
            continue
        lat = safe_float(r.get("centroid_lat"), cfg.GHANA_CENTER_LAT)
        lon = safe_float(r.get("centroid_lon"), cfg.GHANA_CENTER_LON)
        folium.Marker(
            location = [lat, lon + 0.4],
            icon     = folium.DivIcon(
                html=(f'<div style="font-size:9px;color:#dc2626;font-weight:700;'
                      f'white-space:nowrap">⚠ {", ".join(miss[:2])}</div>'),
                icon_size=(220, 18),
            ),
            tooltip=f"{r['region']}: Missing {', '.join(miss)}",
        ).add_to(gap_group)
    gap_group.add_to(m)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend = """
    <div style="position:fixed;bottom:28px;left:28px;z-index:1000;background:#fff;
                padding:12px 16px;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,.2);
                font-size:12px;font-family:sans-serif">
        <b style="font-size:13px">Medical Desert Score (MDS)</b><br><br>
        <span style="color:#991b1b">■</span> Severe Desert (&gt;0.80)<br>
        <span style="color:#c2410c">■</span> Moderate Desert (0.65–0.80)<br>
        <span style="color:#b45309">■</span> Marginal (0.45–0.65)<br>
        <span style="color:#15803d">■</span> Adequate (&lt;0.45)<br>
        <hr style="margin:6px 0;border:0;border-top:1px solid #e5e7eb">
        <span style="color:#dc2626">■</span> Anomaly / Ghost risk<br>
        <span style="color:#1d4ed8">■</span> Hospital &nbsp;
        <span style="color:#15803d">■</span> Clinic &nbsp;
        <span style="color:#7e22ce">■</span> NGO<br>
        <span style="color:#ea580c">■</span> Pharmacy/Other
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))

    title = """
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:1000;
                background:#fff;padding:8px 24px;border-radius:8px;
                box-shadow:0 2px 10px rgba(0,0,0,.2);font-size:14px;font-weight:700;
                font-family:sans-serif">
        🏥 Virtue Foundation Ghana — Medical Desert Analysis (v12)
    </div>
    """
    m.get_root().html.add_child(folium.Element(title))

    folium.LayerControl(collapsed=False).add_to(m)
    return m


print("Building Folium map…")
folium_map = build_medical_desert_map(combined_pd, mds_pd, overlay_pd)
print(f"Map built: {len(folium_map._children)} layers")

# COMMAND ----------
# MAGIC %md ## 8. Part B — Plotly Dashboard

# COMMAND ----------

def build_plotly_dashboard(mds_df: pd.DataFrame, fac_df: pd.DataFrame, overlay_df: pd.DataFrame) -> Dict[str, Any]:
    """
    7 Plotly charts:
    1. MDS bar (all regions)
    2. MDS component breakdown (top 5 underserved)
    3. Critical specialty gap bar (covered vs missing)
    4. Facility type distribution by region
    5. Doctor-to-bed scatter (vs WHO targets)
    6. Anomaly & ghost risk distribution
    7. NGO / volunteer coverage by region
    """
    colour_map = {
        "Severe Desert":   "#b91c1c",
        "Moderate Desert": "#ea580c",
        "Marginal":        "#b45309",
        "Adequate":        "#15803d",
        "Unknown":         "#6b7280",
    }

    # ── Chart 1: MDS bar ──────────────────────────────────────────────────────
    mds_sorted = mds_df.sort_values("medical_desert_score", ascending=True).copy()
    mds_sorted["label_text"] = mds_sorted.apply(
        lambda r: f"{r['desert_label']} (conf={r.get('score_confidence',0):.2f})", axis=1)

    fig1 = px.bar(
        mds_sorted,
        x          = "medical_desert_score",
        y          = "region",
        orientation= "h",
        color      = "desert_label",
        color_discrete_map = colour_map,
        title      = "Blended Medical Desert Score by Region (v12)",
        labels     = {"medical_desert_score": "MDS (0=Adequate → 1=Severe Desert)", "region": ""},
        text       = "label_text",
    )
    fig1.update_layout(height=550, xaxis_range=[0, 1.05], showlegend=True)
    fig1.update_traces(textposition="outside", textfont_size=10)

    # ── Chart 2: MDS component breakdown (top 6 underserved) ─────────────────
    top_n = mds_df.head(6)
    comp_cols   = ["density_component", "specialty_component", "integrity_component", "confidence_component"]
    comp_labels = ["Density Gap", "Specialty Gap", "Integrity Risk", "Data Confidence Gap"]
    comp_colors = ["#1d4ed8", "#dc2626", "#7c3aed", "#ca8a04"]

    fig2 = go.Figure()
    for col, label, color in zip(comp_cols, comp_labels, comp_colors):
        if col in top_n.columns:
            fig2.add_bar(
                x    = top_n["region"],
                y    = top_n[col].apply(lambda v: safe_float(v)),
                name = label,
                marker_color = color,
            )
    fig2.update_layout(
        barmode = "group",
        title   = "MDS Component Breakdown — 6 Most Underserved Regions",
        yaxis   = dict(title="Component Score (1.0 = maximum desert pressure)", range=[0, 1.05]),
        xaxis_tickangle = -30,
        height  = 460,
    )

    # ── Chart 3: Specialty gap ────────────────────────────────────────────────
    spec_df = mds_df[["region", "desert_label", "critical_specialty_gap_count"]].copy()
    spec_df["covered"] = len(cfg.CRITICAL_SPECIALTIES) - spec_df["critical_specialty_gap_count"].fillna(0)
    spec_df = spec_df.sort_values("critical_specialty_gap_count", ascending=False)

    fig3 = go.Figure()
    fig3.add_bar(x=spec_df["region"], y=spec_df["covered"],
                 name="Covered", marker_color="#16a34a")
    fig3.add_bar(x=spec_df["region"], y=spec_df["critical_specialty_gap_count"],
                 name="Missing Critical", marker_color="#dc2626")
    fig3.update_layout(
        barmode="stack",
        title  ="Critical Specialty Coverage by Region",
        xaxis_title="Region", yaxis_title=f"Specialties (of {len(cfg.CRITICAL_SPECIALTIES)} critical)",
        xaxis_tickangle=-45, height=460,
    )

    # ── Chart 4: Facility type distribution ───────────────────────────────────
    type_cols = {
        "Hospitals": "hospital_count",
        "Clinics":   "clinic_count",
        "NGOs":      "ngo_count",
    }
    fig4 = go.Figure()
    colors4 = {"Hospitals": "#1d4ed8", "Clinics": "#15803d", "NGOs": "#7c3aed"}
    type_sorted = mds_df.sort_values("total_facilities", ascending=False)
    for label, col in type_cols.items():
        if col in type_sorted.columns:
            fig4.add_bar(
                x    = type_sorted["region"],
                y    = type_sorted[col].apply(lambda v: safe_float(v)),
                name = label,
                marker_color = colors4[label],
            )
    fig4.update_layout(
        barmode="stack", title="Facility Type Distribution by Region",
        xaxis_tickangle=-45, height=460,
    )

    # ── Chart 5: Doctor-to-bed scatter ────────────────────────────────────────
    fig5 = px.scatter(
        mds_df,
        x         = "doctors_per_100k",
        y         = "beds_per_100k",
        color     = "desert_label",
        color_discrete_map = colour_map,
        size      = "total_facilities",
        size_max  = 35,
        text      = "region",
        hover_data= ["medical_desert_score", "score_confidence", "hospitals_per_100k"],
        title     = "Doctor-to-Bed Ratio by Region (vs WHO Targets)",
        labels    = {
            "doctors_per_100k": "Doctors per 100,000 population",
            "beds_per_100k":    "Beds per 100,000 population",
        },
    )
    # WHO targets: 2.3 doctors/10k = 23/100k; 10 beds/10k = 100/100k
    fig5.add_vline(x=23, line_dash="dash", line_color="#6b7280",
                   annotation_text="WHO doctor target (23/100k)",
                   annotation_position="top right")
    fig5.add_hline(y=100, line_dash="dash", line_color="#6b7280",
                   annotation_text="WHO bed target (100/100k)",
                   annotation_position="right")
    fig5.update_traces(textposition="top center", textfont_size=9)
    fig5.update_layout(height=520)

    # ── Chart 6: Anomaly & ghost risk ─────────────────────────────────────────
    anom_df = mds_df[["region", "total_region_anomalies", "avg_ghost_probability",
                       "desert_label"]].copy()
    anom_df = anom_df.sort_values("total_region_anomalies", ascending=False)

    fig6 = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Total Anomalies by Region", "Avg Ghost Probability by Region"]
    )
    fig6.add_bar(
        x=anom_df["region"], y=anom_df["total_region_anomalies"],
        name="Anomalies", marker_color="#dc2626", row=1, col=1
    )
    fig6.add_bar(
        x=anom_df["region"], y=anom_df["avg_ghost_probability"],
        name="Ghost Prob.", marker_color="#7c3aed", row=1, col=2
    )
    fig6.update_layout(
        title="Data Integrity: Anomalies & Ghost Facility Risk",
        height=460, showlegend=True,
    )
    fig6.update_xaxes(tickangle=-45)

    # ── Chart 7: NGO & volunteer coverage ─────────────────────────────────────
    ngo_df = mds_df[["region", "ngo_count", "volunteer_facilities",
                      "icu_facilities", "emergency_medicine_facilities", "desert_label"]].copy()
    ngo_df = ngo_df.sort_values("ngo_count", ascending=False)

    fig7 = go.Figure()
    fig7.add_bar(x=ngo_df["region"], y=ngo_df["ngo_count"],
                 name="NGOs", marker_color="#0891b2")
    fig7.add_bar(x=ngo_df["region"], y=ngo_df["volunteer_facilities"],
                 name="Accept Volunteers", marker_color="#7c3aed")
    fig7.add_bar(x=ngo_df["region"], y=ngo_df["emergency_medicine_facilities"],
                 name="Emergency Medicine", marker_color="#15803d")
    fig7.add_bar(x=ngo_df["region"], y=ngo_df["icu_facilities"],
                 name="ICU", marker_color="#dc2626")
    fig7.update_layout(
        barmode="group",
        title  ="NGO, Volunteer & Critical Service Coverage by Region",
        xaxis_tickangle=-45, height=480,
    )

    return {
        "mds_bar":             fig1,
        "component_breakdown": fig2,
        "specialty_gap":       fig3,
        "type_distribution":   fig4,
        "doctor_bed_scatter":  fig5,
        "anomaly_integrity":   fig6,
        "ngo_volunteer":       fig7,
    }


print("Building Plotly charts…")
charts = build_plotly_dashboard(mds_pd, combined_pd, overlay_pd)
print(f"Built {len(charts)} charts")
for name, fig in charts.items():
    fig.show()

# COMMAND ----------
# MAGIC %md ## 9. Export to HTML + GeoJSON + CSV

# COMMAND ----------

def save_all_outputs(
    folium_m:   folium.Map,
    charts_d:   Dict,
    mds_df:     pd.DataFrame,
    overlay_df: pd.DataFrame,
    fac_df:     pd.DataFrame,
) -> List[str]:
    """Write all outputs to both Workspace and Volume paths."""
    written: List[str] = []
    region_mds_map   = dict(zip(mds_df["region"], mds_df["medical_desert_score"]))

    for base in [cfg.OUTPUT_WORKSPACE, cfg.OUTPUT_VOLUME]:
        Path(base).mkdir(parents=True, exist_ok=True)

        # Folium map
        try:
            mp = str(Path(base) / "medical_desert_map.html")
            folium_m.save(mp)
            written.append(mp)
            print(f"  Map HTML       : {mp}")
        except Exception as e:
            print(f"  Map save failed ({base}): {e}")

        # Plotly charts
        for name, fig in charts_d.items():
            try:
                cp = str(Path(base) / f"chart_{name}.html")
                fig.write_html(cp, include_plotlyjs="cdn", full_html=True)
                written.append(cp)
                print(f"  Chart          : {cp}")
            except Exception as e:
                print(f"  Chart {name} failed: {e}")

        # GeoJSON (facilities)
        try:
            features = []
            geo_fac = fac_df[
                (fac_df["latitude"]  != 0) & fac_df["latitude"].notna() &
                (fac_df["longitude"] != 0) & fac_df["longitude"].notna()
            ]
            for _, row in geo_fac.iterrows():
                lat = safe_float(row.get("latitude"),  0.0)
                lon = safe_float(row.get("longitude"), 0.0)
                if lat == 0 and lon == 0:
                    continue
                region = safe_str(row.get("region_normalised", "Unknown"))
                mds_s  = safe_float(
                    row.get("facility_desert_overlay")
                    or region_mds_map.get(region)
                    or row.get("medical_desert_score"),
                    0.5
                )
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": [lon, lat]},
                    "properties": {
                        "id":              safe_str(row.get("unique_id")),
                        "name":            safe_str(row.get("name")),
                        "type":            safe_str(row.get("facility_type_clean")),
                        "tier":            safe_str(row.get("facility_tier_label")),
                        "region":          region,
                        "city":            safe_str(row.get("city_clean")),
                        "desert_score":    round(mds_s, 4),
                        "desert_label":    mds_label(mds_s),
                        "has_icu":         safe_bool(row.get("has_icu")),
                        "has_surgery":     safe_bool(row.get("has_surgery")),
                        "has_emergency":   safe_bool(row.get("has_emergency_medicine")),
                        "has_obstetrics":  safe_bool(row.get("has_obstetrics")),
                        "accepts_vol":     safe_bool(row.get("accepts_volunteers_bool")),
                        "is_ngo":          safe_bool(row.get("is_ngo")),
                        "doctors":         safe_int(row.get("number_doctors_int")),
                        "beds":            safe_int(row.get("capacity_int")),
                        "completeness":    round(safe_float(row.get("data_completeness_score")), 3),
                        "rag_quality":     round(safe_float(row.get("rag_quality_score")), 3),
                        "ghost_prob":      round(safe_float(row.get("ghost_probability_score")), 3),
                        "has_anomaly":     not safe_bool(row.get("capability_is_valid", True)),
                        "em_readiness":    round(safe_float(row.get("emergency_readiness_score")), 3),
                        "critical_care":   round(safe_float(row.get("critical_care_score")), 3),
                        "color":           mds_color(mds_s),
                    },
                })
            gp = str(Path(base) / "ghana_facilities.geojson")
            with open(gp, "w", encoding="utf-8") as f:
                json.dump({"type": "FeatureCollection", "features": features}, f)
            written.append(gp)
            print(f"  GeoJSON        : {gp}  ({len(features):,} features)")
        except Exception as e:
            print(f"  GeoJSON failed: {e}")

        # Regional MDS CSV
        try:
            cp = str(Path(base) / "regional_desert_scores.csv")
            mds_df.to_csv(cp, index=False)
            written.append(cp)
            print(f"  MDS CSV        : {cp}")
        except Exception as e:
            print(f"  MDS CSV failed: {e}")

        # Facility overlay CSV
        try:
            op = str(Path(base) / "facility_desert_overlay.csv")
            overlay_df.head(200).to_csv(op, index=False)
            written.append(op)
            print(f"  Overlay CSV    : {op}")
        except Exception as e:
            print(f"  Overlay CSV failed: {e}")

        # Planner action cards JSON
        try:
            cards = []
            for _, r in mds_df.iterrows():
                cards.append({
                    "region":        r["region"],
                    "desert_label":  r["desert_label"],
                    "mds":           round(safe_float(r.get("medical_desert_score")), 4),
                    "confidence":    round(safe_float(r.get("score_confidence")), 4),
                    "rationale":     safe_str(r.get("score_rationale")),
                    "actions":       ensure_list(r.get("recommended_actions")),
                    "missing_specs": ensure_list(r.get("missing_critical_specialties")),
                    "ngo_count":     safe_int(r.get("ngo_count")),
                    "volunteer_fac": safe_int(r.get("volunteer_facilities")),
                })
            jp = str(Path(base) / "planner_action_cards.json")
            with open(jp, "w", encoding="utf-8") as f:
                json.dump(cards, f, ensure_ascii=False, indent=2)
            written.append(jp)
            print(f"  Planner cards  : {jp}")
        except Exception as e:
            print(f"  Planner cards failed: {e}")

    return written


print("\nSaving output files…")
written = save_all_outputs(folium_map, charts, mds_pd, overlay_pd, combined_pd)
print(f"\n✅  Total files written: {len(written)}")

# COMMAND ----------
# MAGIC %md ## 10. Render Map in Notebook

# COMMAND ----------

ws_map = str(Path(cfg.OUTPUT_WORKSPACE) / "medical_desert_map.html")
try:
    with open(ws_map, "r", encoding="utf-8") as f:
        displayHTML(f.read())
    print(f"✅  Map rendered ({Path(ws_map).stat().st_size / 1e6:.2f} MB)")
except Exception as e:
    print(f"displayHTML skipped: {e}")
    print(f"Open map at: {ws_map}")

# COMMAND ----------
# MAGIC %md ## 11. MLflow Logging

# COMMAND ----------

try:
    mlflow.set_experiment(cfg.MLFLOW_EXP)
    with mlflow.start_run(run_name="07_medical_desert_scoring") as run:
        # Parameters
        mlflow.log_param("schema_version",   cfg.SCHEMA_VER)
        mlflow.log_param("mds_method",       "blended_v12")
        mlflow.log_param("w_base",           cfg.W_BASE)
        mlflow.log_param("w_specialty",      cfg.W_SPECIALTY)
        mlflow.log_param("w_integrity",      cfg.W_INTEGRITY)
        mlflow.log_param("w_confidence",     cfg.W_CONFIDENCE)
        mlflow.log_param("regional_table",   cfg.REGIONAL_TABLE)
        mlflow.log_param("facilities_table", cfg.FACILITIES_TABLE)
        # Region-level metrics
        mlflow.log_metric("regions_scored",            len(mds_pd))
        mlflow.log_metric("severe_desert_regions",
                          int((mds_pd["desert_label"] == "Severe Desert").sum()))
        mlflow.log_metric("moderate_desert_regions",
                          int((mds_pd["desert_label"] == "Moderate Desert").sum()))
        mlflow.log_metric("marginal_regions",
                          int((mds_pd["desert_label"] == "Marginal").sum()))
        mlflow.log_metric("adequate_regions",
                          int((mds_pd["desert_label"] == "Adequate").sum()))
        mlflow.log_metric("avg_mds",        round(float(mds_pd["medical_desert_score"].mean()), 4))
        mlflow.log_metric("max_mds",        round(float(mds_pd["medical_desert_score"].max()), 4))
        mlflow.log_metric("avg_confidence", round(float(mds_pd.get("score_confidence",
                                                                    pd.Series([0])).mean()), 4))
        # Facility metrics
        mlflow.log_metric("total_facilities_combined", len(combined_pd))
        mlflow.log_metric("facilities_mapped",
                          int(((combined_pd["latitude"] != 0) & (combined_pd["longitude"] != 0)).sum()))
        mlflow.log_metric("overlay_rows",   len(overlay_pd))
        mlflow.log_metric("charts_built",   len(charts))
        mlflow.log_metric("files_written",  len(written))
        # Artifacts
        mlflow.log_dict(mds_pd.to_dict("records"), "regional_mds_scores.json")
        ws_csv = str(Path(cfg.OUTPUT_WORKSPACE) / "regional_desert_scores.csv")
        if Path(ws_csv).exists():
            mlflow.log_artifact(ws_csv)
        ws_cards = str(Path(cfg.OUTPUT_WORKSPACE) / "planner_action_cards.json")
        if Path(ws_cards).exists():
            mlflow.log_artifact(ws_cards)
        print(f"MLflow run: {run.info.run_id}")
except Exception as e:
    print(f"MLflow skipped: {e}")

# COMMAND ----------
# MAGIC %md ## 12. Summary

# COMMAND ----------

print(f"\n{'='*70}")
print(f"✅  NOTEBOOK 07 — Medical Desert Scoring v12 — COMPLETE")
print(f"{'='*70}")

print(f"\n  Schema version       : {cfg.SCHEMA_VER}")
print(f"  Regions scored       : {len(mds_pd)}")
dist = Counter(mds_pd["desert_label"])
for label in ["Severe Desert", "Moderate Desert", "Marginal", "Adequate"]:
    print(f"    {label:<20}: {dist.get(label, 0)}")
print(f"  Avg MDS              : {mds_pd['medical_desert_score'].mean():.3f}")
print(f"  Avg score confidence : {mds_pd.get('score_confidence', pd.Series([0])).mean():.3f}")
print(f"  Facilities in overlay: {len(overlay_pd):,}")
print(f"  Charts built         : {len(charts)}")
print(f"  Files written        : {len(written)}")

print(f"\n  Top 5 most underserved regions:")
for _, r in mds_pd.head(5).iterrows():
    ec = r.get("emergency_gap_score", 0.0)
    ic = r.get("icu_gap_score", 0.0)
    print(f"    {r['region']:<22} MDS={r['medical_desert_score']:.3f}  "
          f"[{r['desert_label']}]  "
          f"emGap={ec:.2f}  icuGap={ic:.2f}")

print(f"\n  Delta tables:")
print(f"    {cfg.MDS_TABLE}")
print(f"    {cfg.OVERLAY_TABLE}")

print(f"\n  Key output files:")
for p in written[:10]:
    print(f"    {p}")


In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_medical_desert_scores

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_medical_desert_scores